# GroundingDINO + SAM Pipeline

This tutorial demonstrates the full detect-then-segment pipeline:
1. **GroundingDINO** detects objects from a text prompt (zero-shot).
2. **SAM** segments each detected bounding box.

**Models:** `grounding-dino-swin-t` + `sam-vit-b`  
**License:** Apache-2.0 (both models)  
**VisionServeX API:** `vsx.pipeline('gdino+sam', ...)`

In [ ]:
# Cell 2: Step 1 — GroundingDINO detection
import subprocess, sys
subprocess.check_call([sys.executable, '-m', 'pip', 'install', 'visionservex', '-q'])

import visionservex as vsx
import numpy as np
from visionservex.grounding_dino import run_grounding_dino_detect

# Synthetic test image
image = np.random.randint(0, 255, (480, 640, 3), dtype=np.uint8)
text_prompt = 'a cat . a dog . a chair .'

detect_result = run_grounding_dino_detect(
    model_id='grounding-dino-swin-t',
    image=image,
    text_prompt=text_prompt,
    box_threshold=0.30,
    text_threshold=0.25
)

print(f'[Step 1] GroundingDINO detection')
print(f'  Status       : {detect_result["status"]}')
print(f'  Detections   : {detect_result.get("num_detections", 0)}')
print(f'  Latency ms   : {detect_result.get("latency_ms", "N/A")}')
for b in detect_result.get('boxes', []):
    print(f'  [{b["label"]:12s}] score={b["score"]:.3f} box={b["xyxy"]}')

In [ ]:
# Cell 3: Step 2 — SAM segment each detected box
from visionservex.sam import run_sam_box_batch

boxes = [b['xyxy'] for b in detect_result.get('boxes', [])]

if boxes:
    seg_result = run_sam_box_batch(
        model_id='sam-vit-b',
        image=image,
        boxes=boxes
    )
    print(f'[Step 2] SAM segmentation')
    print(f'  Status       : {seg_result["status"]}')
    print(f'  Masks count  : {seg_result.get("num_masks", 0)}')
    print(f'  Latency ms   : {seg_result.get("latency_ms", "N/A")}')
else:
    seg_result = {'status': 'skipped', 'num_masks': 0, 'masks': []}
    print('[Step 2] SAM skipped — no boxes from detection step')

In [ ]:
# Cell 4: Full pipeline result visualization
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 5))

# Input image with detected boxes
axes[0].imshow(image)
colors = plt.cm.Set1.colors
for i, b in enumerate(detect_result.get('boxes', [])):
    x1,y1,x2,y2 = b['xyxy']
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=2,
                               edgecolor=colors[i%len(colors)], facecolor='none')
    axes[0].add_patch(rect)
    axes[0].text(x1, y1-4, f'{b["label"]} {b["score"]:.2f}', color=colors[i%len(colors)], fontsize=8)
axes[0].set_title(f'Step 1: GD Detections ({detect_result.get("num_detections",0)})')

# SAM masks overlay
mask_overlay = np.zeros((*image.shape[:2], 4), dtype=np.float32)
for i, m in enumerate(seg_result.get('masks', [])):
    if m is not None:
        c = colors[i % len(colors)]
        mask_overlay[m.squeeze().astype(bool)] = [*c[:3], 0.5]
axes[1].imshow(image)
axes[1].imshow(mask_overlay)
axes[1].set_title(f'Step 2: SAM Masks ({seg_result.get("num_masks",0)})')

# Combined
axes[2].imshow(image)
axes[2].imshow(mask_overlay)
for i, b in enumerate(detect_result.get('boxes', [])):
    x1,y1,x2,y2 = b['xyxy']
    rect = patches.Rectangle((x1,y1), x2-x1, y2-y1, linewidth=1,
                               edgecolor=colors[i%len(colors)], facecolor='none', linestyle='--')
    axes[2].add_patch(rect)
axes[2].set_title('Combined: GD + SAM')

plt.tight_layout()
plt.savefig('/tmp/gdino_sam_pipeline.png', dpi=80)
plt.show()
print('Pipeline result saved to /tmp/gdino_sam_pipeline.png')

In [ ]:
# Cell 5: VSX pipeline() API — high-level one-call interface
pipeline_result = vsx.pipeline(
    name='gdino+sam',
    image=image,
    text_prompt=text_prompt,
    detector='grounding-dino-swin-t',
    segmentor='sam-vit-b',
    box_threshold=0.30
)

print(f'vsx.pipeline status     : {pipeline_result["status"]}')
print(f'Detected objects        : {pipeline_result.get("num_detections", 0)}')
print(f'Segmented masks         : {pipeline_result.get("num_masks", 0)}')
print(f'Total latency ms        : {pipeline_result.get("total_latency_ms", "N/A")}')
print(f'  Detect ms             : {pipeline_result.get("detect_latency_ms", "N/A")}')
print(f'  Segment ms            : {pipeline_result.get("segment_latency_ms", "N/A")}')

## Upgrade Paths

| Detector | Segmentor | Requirements | Speed |
|---|---|---|---|
| grounding-dino-swin-t | sam-vit-b | Apache-2.0, no auth | fast |
| grounding-dino-swin-t | mobilesam | Apache-2.0, no auth | fastest |
| grounding-dino-swin-t | sam2.1-hiera-tiny | Apache-2.0 + sam2 pkg | balanced |
| grounding-dino-original-swin-t | sam-vit-b | HF auth required | accurate |
| grounding-dino-swin-t | sam3 | HF auth required | best quality |

## BYOT Pipeline Status

To use gated models in the pipeline, provide your HF token:

```python
import os
os.environ['HF_TOKEN'] = 'hf_your_token_here'

# Then pull the gated model
# visionservex pull grounding-dino-original-swin-t
# visionservex pull sam3

# Run the premium pipeline
result = vsx.pipeline(
    name='gdino+sam',
    detector='grounding-dino-original-swin-t',
    segmentor='sam3',
    image=image,
    text_prompt='a cat . a dog .'
)
```

## Next Steps

- For SAM ONNX export, see `01_sam_onnx_runtime.ipynb`.
- For DINOv3 embedding, see `05_dinov3_license_aware.ipynb`.
- For SAM3 auth workflow, see `04_sam3_byot_auth.ipynb`.